# D10 · Modelos basados en árboles — Lección

**Bootcamp de Datos para Funcionarios Públicos · Formación Pública**
Línea B · *Data Scientist* · Semana 11 · Se ejecuta en Google Colab.\n\n> 🚀 **Cómo se trabaja:** lee, ejecuta cada celda con **▶︎** (o `Shift`+`Enter`) y completa los `TODO`. Cada ejercicio termina en una **celda de chequeo** que muestra ✅ si está bien o una pista si todavía no.

---

## ¿Qué vas a lograr?

En D9 entrenaste un modelo de regresión lineal para predecir el monto de compras públicas. Hoy usarás
modelos mucho más potentes: **árboles de decisión** y **bosques aleatorios** (*Random Forests*).
Aprenderás a pasar de una sola feature a múltiples features, a visualizar cómo toma decisiones un
árbol y verás de forma gráfica cómo se produce y se controla el enemigo número uno: el **sobreajuste**.

**Competencia de salida:** entrenar y evaluar regresores basados en árboles de decisión y ensembles,
controlar su complejidad usando hiperparámetros (profundidad), e interpretar la importancia de las features.

**Dato real:** cantidad de artículos, tamaño numérico del proveedor (`tamano_num`) y monto total de compras públicas de alimentos.

**Entregable:** que las **4 celdas de chequeo** muestren ✅.


## Por qué seguimos con el mismo dato (y le sumamos una variable)

Mantenemos el dominio de compras públicas de alimentos a propósito: al no cambiar el problema,
puedes comparar **de forma justa** la regresión lineal de D9 con los árboles de hoy, y concentrarte
en lo nuevo —el modelo— sin distraerte con un dataset distinto. Además, ahora usaremos **dos features**
a la vez: la `cantidad` de artículos y el tamaño del proveedor ordinarizado (`tamano_num` de 1 a 4, donde
1 es Micro y 4 es Grande).


In [ ]:
# ── Preparación del entorno (ejecuta esta celda primero) ──────────────────────
import os
import urllib.request
import pandas as pd
import matplotlib.pyplot as plt

# Si el archivo no existe localmente (ej: en Colab), intentamos descargarlo desde GitHub
if not os.path.exists("compras_ml.csv"):
    try:
        url = "https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/B3-modelos-de-arboles/compras_ml.csv"  # Reemplazar por la URL real del repositorio al publicar
        urllib.request.urlretrieve(url, "compras_ml.csv")
        print("compras_ml.csv descargado automáticamente.")
    except Exception:
        print("Aviso: No se pudo descargar compras_ml.csv automáticamente. Si estás en Colab, asegúrate de subirlo manualmente.")

df = pd.read_csv("compras_ml.csv")
print(f"{len(df)} compras cargadas.")
df.head()


## 1. ¿Qué es un árbol de decisión?

Un árbol de decisión predice haciendo una **secuencia de preguntas sí/no** sobre las *features*,
hasta llegar a una respuesta. Para predecir el monto total de la compra podría preguntar:
*¿la cantidad es mayor a 100 artículos?* → si sí, *¿el proveedor es de tamaño Mediano o Grande (`tamano_num` >= 3)?* → …
y al final entrega el monto total estimado.

> 🧠 **Analogía.** Es como jugar al "¿Quién es quién?" o seguir una guía de trámites del Estado:
> "Si es licitación pública, pase a ventanilla A; si es trato directo, si el monto es < 100 UTM..."

Miremos cómo se ve un árbol pequeño entrenado con nuestros datos:


In [ ]:
# (ilustracion) Asi "piensa" un arbol de decision de profundidad 2.
from sklearn.tree import DecisionTreeRegressor, plot_tree
arbol_demo = DecisionTreeRegressor(max_depth=2, random_state=42).fit(
    df[["cantidad", "tamano_num"]], df["monto_total"])

plt.figure(figsize=(12, 5))
plot_tree(arbol_demo, feature_names=["cantidad", "tamano_num"],
          filled=True, rounded=True, fontsize=9)
plt.title("Visualización de un árbol de decisión de profundidad 2")
plt.show()


### ✍️ Ejercicio 1 — Preparar los datos (dos features)
Define `X` con **las dos** columnas `cantidad` y `tamano_num`, e `y` con `monto_total`, y
divide con `train_test_split` (`test_size=0.3`, `random_state=42`). Completa la celda.


In [ ]:
X = df[["cantidad", "tamano_num"]]
y = df["monto_total"]

from sklearn.model_selection import train_test_split
# TODO: divide con test_size=0.3 y random_state=42 (reemplaza la linea de abajo)
X_train, X_test, y_train, y_test = X, X, y, y   # <-- PLACEHOLDER: aun no divide nada

print("Entrenamiento:", len(X_train), "| Prueba:", len(X_test))


In [ ]:
# ── Celda de chequeo · Ejercicio 1 ───────────────────────────────────────────
try:
    assert list(X_train.columns) == ["cantidad", "tamano_num"], \
        "X debe tener las DOS columnas: cantidad y tamano_num."
    assert len(X_train) == 5188 and len(X_test) == 2224, \
        f"Con test_size=0.3 deben quedar 5188 de entrenamiento y 2224 de prueba; tienes {len(X_train)} y {len(X_test)}."
    print("✅ Correcto: preparaste X, y, y dividiste en entrenamiento y prueba.")
except Exception as e:
    print("❌ Aun no. Revisa tu seleccion de columnas y division, y vuelve a intentar.")
    print("   Pista:", e)


## 2. Entrenar un árbol

Igual que en D9, en dos pasos. El parámetro **`max_depth`** limita cuántas preguntas seguidas
puede hacer el árbol (su "profundidad"). Empezamos con un árbol moderado, `max_depth=3`:

```python
from sklearn.tree import DecisionTreeRegressor
arbol = DecisionTreeRegressor(max_depth=3, random_state=42)
arbol.fit(X_train, y_train)
y_pred_arbol = arbol.predict(X_test)
```

> 💡 **`random_state=42`** vuelve el resultado reproducible (el árbol toma algunas decisiones con
> azar; fijar la semilla hace que tú y yo obtengamos el mismo árbol).

> ⚠️ **Errores típicos**
> - **No fijar `max_depth`** y sorprenderse luego con el sobreajuste (lo veremos en la sección 4).
> - **Olvidar `.fit` antes de `.predict`**, igual que en D9.


### ✍️ Ejercicio 2 — Entrenar y evaluar el árbol
Entrena `arbol` con los datos de entrenamiento, predice sobre `X_test` y calcula el MAE en prueba,
guardándolo en `mae_arbol`. Completa la celda.


In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

arbol = DecisionTreeRegressor(max_depth=3, random_state=42)
# TODO: entrena (arbol.fit), predice (arbol.predict sobre X_test) y calcula el MAE
#       guardandolo en mae_arbol.
mae_arbol = 99.0   # <-- PLACEHOLDER

print(f"MAE del arbol (profundidad 3) en prueba: {mae_arbol:.2f} CLP")


In [ ]:
# ── Celda de chequeo · Ejercicio 2 ───────────────────────────────────────────
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error
try:
    assert hasattr(arbol, "tree_"), "Aun no entrenaste el arbol: falta arbol.fit(X_train, y_train)."
    ref = DecisionTreeRegressor(max_depth=3, random_state=42).fit(X_train, y_train)
    mae_ref = mean_absolute_error(y_test, ref.predict(X_test))
    assert abs(float(mae_arbol) - mae_ref) < 0.05, f"Tu MAE deberia ser ~{mae_ref:.2f} CLP."
    print(f"✅ Correcto. El arbol de profundidad 3 logro un MAE en prueba de ~{mae_ref:.2f} CLP.")
except Exception as e:
    print("❌ Aun no. Revisa tu entrenamiento y calculo de MAE, y vuelve a intentar.")
    print("   Pista:", e)


## 3. El sobreajuste, ahora visible

En D9 definimos el **sobreajuste**: cuando un modelo memoriza el entrenamiento en vez de aprender
el patrón, y por eso falla con datos nuevos. Los árboles lo muestran de manera espectacular. Si a
un árbol **no le pones límite de profundidad**, seguirá haciendo preguntas hasta separar individualmente
las compras sucias o raras de tu entrenamiento. El error en entrenamiento será cero, pero en prueba
se disparará.


In [ ]:
# (ilustracion) El sobreajuste crece con la profundidad del arbol.
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

Xi = df[["cantidad", "tamano_num"]]; yi = df["monto_total"]
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(Xi, yi, test_size=0.3, random_state=42)

profundidades = list(range(1, 11))
errores_train, errores_test = [], []

for d in profundidades:
    m = DecisionTreeRegressor(max_depth=d, random_state=42).fit(Xi_tr, yi_tr)
    errores_train.append(mean_absolute_error(yi_tr, m.predict(Xi_tr)))
    errores_test.append(mean_absolute_error(yi_te, m.predict(Xi_te)))

plt.figure(figsize=(7, 4))
plt.plot(profundidades, errores_train, label="Entrenamiento (memorizacion)", color="blue", marker="o")
plt.plot(profundidades, errores_test, label="Prueba (generalizacion)", color="red", marker="s")
plt.xlabel("Profundidad del arbol")
plt.ylabel("MAE (en CLP)")
plt.title("MAE vs Profundidad: el punto de mejor generalizacion")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


> 🧠 **La idea clave.** Un modelo más complejo no es "mejor". El objetivo no es acertar en los
> datos que ya tenemos, sino **generalizar** a los que vendrán. Controlar la complejidad (aquí,
> limitar la profundidad) es parte central del oficio.

### ✍️ Ejercicio 3 — Ver el sobreajuste con tus manos
Entrena un `arbol_profundo` **sin** límite de profundidad (`DecisionTreeRegressor(random_state=42)`)
y calcula su MAE en **entrenamiento** (`mae_train`) y en **prueba** (`mae_test`). Compara los dos
números: deberías ver el entrenamiento casi en 0 y la prueba bastante más alta. Completa la celda.


In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

# Sin max_depth, el arbol crece sin limite y memoriza el entrenamiento.
arbol_profundo = DecisionTreeRegressor(random_state=42)
# TODO: entrena arbol_profundo y calcula el MAE en ENTRENAMIENTO (mae_train)
#       y en PRUEBA (mae_test).
mae_train = 99.0   # <-- PLACEHOLDER
mae_test  = 99.0   # <-- PLACEHOLDER

print(f"Arbol profundo - MAE en entrenamiento: {mae_train:.2f} CLP | MAE en prueba: {mae_test:.2f} CLP")


In [ ]:
# ── Celda de chequeo · Ejercicio 3 ───────────────────────────────────────────
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error
try:
    assert hasattr(arbol_profundo, "tree_"), "Primero entrena el arbol profundo (.fit)."
    ref = DecisionTreeRegressor(random_state=42).fit(X_train, y_train)
    mt = mean_absolute_error(y_train, ref.predict(X_train))
    mte = mean_absolute_error(y_test, ref.predict(X_test))
    assert abs(float(mae_train) - mt) < 0.05 and abs(float(mae_test) - mte) < 0.05, \
        f"Tus MAEs no coinciden con los del modelo entrenado. Deberian ser train ~{mt:.2f} y test ~{mte:.2f} CLP."
    print(f"✅ Correcto. MAE entrenamiento: {mae_train:.2f} CLP | MAE prueba: {mae_test:.2f} CLP.")
    print("   ¡Viste la brecha! El error de entrenamiento cayo, pero el de prueba subio.")
    print("   Esa enorme diferencia es la firma del sobreajuste (overfitting).")
except Exception as e:
    print("❌ Aun no. Revisa tu entrenamiento y evaluacion del arbol profundo.")
    print("   Pista:", e)


## 4. Bosques aleatorios: la sabiduría de muchos árboles

Un solo árbol es inestable y propenso a sobreajustar. La solución más usada es el **bosque
aleatorio** (*random forest*): entrenar **muchos** árboles, cada uno sobre una porción distinta de
los datos, y **promediar** sus predicciones.

> 🧠 **Analogía.** Es la "sabiduría de la multitud": una sola persona puede equivocarse feo, pero
> el **promedio de muchas opiniones independientes** suele acertar mejor y ser más estable. Cada
> árbol aporta su voto; el bosque promedia.

Se usa idéntico:

```python
from sklearn.ensemble import RandomForestRegressor
bosque = RandomForestRegressor(n_estimators=100, random_state=42)
bosque.fit(X_train, y_train)
```

Y trae un regalo muy útil para el sector público: la **importancia de variables**
(`bosque.feature_importances_`), que indica **cuánto pesó cada *feature*** en las predicciones.
Saber qué variable manda es a veces más valioso que la predicción misma.

> 📌 **Honestidad estadística.** En este dato, casi una recta, el bosque no necesariamente le gana
> a la regresión lineal de D9: los árboles brillan más cuando las relaciones son **complejas o no
> lineales**. Lo que siempre ganas con ellos es robustez y la lectura de importancias.


### ✍️ Ejercicio 4 — Entrenar un bosque y leer las importancias
Entrena un `RandomForestRegressor(n_estimators=100, random_state=42)`, calcula su MAE en prueba
(`mae_bosque`) y obtén `feature_top`, el nombre de la variable más importante. Completa la celda.


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import pandas as pd

bosque = RandomForestRegressor(n_estimators=100, random_state=42)
# TODO: 1) entrena el bosque (.fit)
#       2) calcula mae_bosque (MAE en prueba)
#       3) obten feature_top = la variable mas importante
#          Pista: pd.Series(bosque.feature_importances_, index=X_train.columns).idxmax()
mae_bosque = 99.0   # <-- PLACEHOLDER
feature_top = "ninguna" # <-- PLACEHOLDER

print(f"MAE del Bosque Aleatorio en prueba: {mae_bosque:.2f} CLP")
print(f"La feature mas influyente para predecir el monto es: {feature_top}")


In [ ]:
# ── Celda de chequeo · Ejercicio 4 ───────────────────────────────────────────
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
try:
    assert hasattr(bosque, "estimators_"), "Aun no entrenaste el bosque: falta bosque.fit(X_train, y_train)."
    ref = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train, y_train)
    mb = mean_absolute_error(y_test, ref.predict(X_test))
    assert abs(float(mae_bosque) - mb) < 0.05, f"Tu MAE deberia ser ~{mb:.2f} CLP."
    assert feature_top == "cantidad", "La variable mas importante deberia ser 'cantidad'."
    print(f"✅ Correcto. El bosque redujo el MAE a ~{mb:.2f} CLP y la variable mas importante es '{feature_top}'.")
    print("   ¡Felicitaciones! Has completado el modulo M10 y dominas arboles y ensembles. 🌲🌲")
except Exception as e:
    print("❌ Aun no. Revisa tu Random Forest y calculos.")
    print("   Pista:", e)


## 5. Cierre

Sumaste a tu caja de herramientas una familia de modelos potente y muy usada:

1. **Árboles de decisión:** predicen con una secuencia de preguntas sí/no; capturan patrones que no
   son una recta.
2. **Sobreajuste, ahora visible:** sin límite de profundidad, el árbol memoriza (error de
   entrenamiento cero) y generaliza mal. Se controla con el hiperparámetro `max_depth`.
3. **Random Forest (Bosque Aleatorio):** un ensemble de muchos árboles entrenados con subconjuntos
   al azar de los datos. Reduce el sobreajuste de forma robusta.
4. **Importancia de features:** permite saber cuál variable tiene más peso en la predicción.

En **D11 · Clasificación y clustering** daremos un paso clave: pasaremos de predecir montos
(regresión) a predecir si un proveedor es Grande o no (clasificación), y agruparemos compras
similares sin conocer etiquetas de antemano (clustering).

### Mini-glosario
- **Árbol de decisión:** modelo jerárquico basado en ramificaciones sí/no.
- **Bosque aleatorio:** ensemble de árboles que promedia sus respuestas.
- **Profundidad (`max_depth`):** hiperparámetro que limita el tamaño del árbol para evitar sobreajuste.
- **Importancia de features:** peso que tiene cada variable en las divisiones del bosque.

---
*Fuente de datos: Dirección de Compras y Contratación Pública (ChileCompra / MercadoPúblico).*
*Contenido bajo licencia CC BY 4.0 · Formación Pública.*
